# 天气预报代理
接下来，构建一个实用的天气预报代理，展示关键的生产概念：

1. 详细的系统提示 以获得更好的代理行为
2. 创建工具 与外部数据集成
3. 模型配置 以确保响应一致性
4. 结构化输出 以获得可预测的结果
5. 对话记忆 以实现类似聊天的交互
6. 创建并运行代理 构建一个功能完整的代理

## 1. 系统提示（Prompt）


In [3]:
SYSTEM_PROMPT = """你是一位擅长用双关语表达的专家天气预报员。
你可以使用两个工具：
- get_weather_for_location：用于获取特定地点的天气
- get_user_location：用于获取用户的位置
如果用户询问天气，请确保你知道具体位置。如果从问题中可以判断他们指的是自己所在的位置，请使用 get_user_location 工具来查找他们的位置。"""

## 2. 创建工具（Tools）
* 概念引入
运行时（runtime）：运行时 = 函数执行那一刻，它所在的环境 + 拿到的资源
比如：你运行工具，工具需要知道：谁调用的？什么权限？什么会话？这些信息在工具被调用时才存在
所以叫运行时信息

ToolRuntime = 工具运行时环境，给工具函数用的 “工具箱 + 上下文传送器”。
里面装着：
你定义的 Context（user_id 等）日志，回调，运行配置，权限信息

In [5]:
from dataclasses import dataclass

from langchain.tools import ToolRuntime, tool


@tool
def get_weather_for_location(city: str) -> str:
    """获取指定城市的天气。"""
    return f"{city}总是阳光明媚！"


@dataclass  # 不用写 init，会自动生成
class Context:
    """
    自定义运行时上下文模式。
    模拟 “上下文 / 会话信息”,不是用户输入的参数，而是系统层面的上下文,
    平时的工具只能传参数def get_weather(city: str)只能传 city,
    真实工程里，工具还需要知道：
    {
        user_id
        会话 ID
        权限
        请求来源
        设备信息
        环境变量
    }
    """

    user_id: str


@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """
    根据用户 ID 获取用户信息。
    """
    user_id = runtime.context.user_id  # 安全获取
    return "Shenzhen" if user_id == "1" else "Beijing"

## 3. 定义响应格式（Response）


In [7]:
@dataclass
class ResponseFormat:
    """代理的响应模式。"""
    # 带双关语的回应（始终必需）
    punny_response: str
    # 天气的任何有趣信息（如果有）
    weather_conditions: str | None = None

## 4. 添加记忆（Memory）
向 Agent（代理） 添加记忆，以在多次交互中保持状态。
这允许 Agent （代理）记住之前的对话和上下文。
在生产环境中，请使用持久化的检查点保存到数据库。

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

## 5. 创建并运行 Agent
将所有组件组装成代理并运行它

In [12]:
from dataclasses import asdict
import json
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model = init_chat_model(
    model="qwen-plus",  # 模型名称可自定义
    model_provider="openai",  # 如果是Langchain不支持的模型类型，需要指定模型提供者，虽然Langchain不支持阿里千问，但是千问兼容openai
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

# print(type(model)) 

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer,
)

# ? 作用是啥？
config = {"configurable":{"thread_id":"1"}}

response = agent.invoke(
    {
        "messages":[
            HumanMessage(content="外面天气怎么样？")
        ]
    },
    config=config,
    context=Context(user_id="1") # 作用是啥？
)

obj=response['structured_response']
print(json.dumps(asdict(obj), indent=2, ensure_ascii=False))

Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


{
  "punny_response": "深圳的天气真是‘深’得我心，‘圳’在阳光里！",
  "weather_conditions": "Shenzhen总是阳光明媚！"
}
